In [ ]:
pip install requests

## Q1. Running Elastic

In [1]:
!curl localhost:9200

{
  "name" : "eb363035b330",
  "cluster_name" : "docker-cluster",
  "cluster_uuid" : "FaccVoeSS5e6WMqIialyXQ",
  "version" : {
    "number" : "8.17.6",
    "build_flavor" : "default",
    "build_type" : "docker",
    "build_hash" : "dbcbbbd0bc4924cfeb28929dc05d82d662c527b7",
    "build_date" : "2025-04-30T14:07:12.231372970Z",
    "build_snapshot" : false,
    "lucene_version" : "9.12.0",
    "minimum_wire_compatibility_version" : "7.17.0",
    "minimum_index_compatibility_version" : "7.0.0"
  },
  "tagline" : "You Know, for Search"
}


In [2]:
!curl https://refactored-tribble-gx5wr5vgvw3p47v-9200.app.github.dev/

{
  "name" : "eb363035b330",
  "cluster_name" : "docker-cluster",
  "cluster_uuid" : "FaccVoeSS5e6WMqIialyXQ",
  "version" : {
    "number" : "8.17.6",
    "build_flavor" : "default",
    "build_type" : "docker",
    "build_hash" : "dbcbbbd0bc4924cfeb28929dc05d82d662c527b7",
    "build_date" : "2025-04-30T14:07:12.231372970Z",
    "build_snapshot" : false,
    "lucene_version" : "9.12.0",
    "minimum_wire_compatibility_version" : "7.17.0",
    "minimum_index_compatibility_version" : "7.0.0"
  },
  "tagline" : "You Know, for Search"
}


In [3]:
import requests 

docs_url = 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/documents.json?raw=1'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

## Q2. Indexing the data

In [ ]:
!pip install "elasticsearch<9.0.0,>=8.0.0"

In [5]:

from elasticsearch import Elasticsearch
client = Elasticsearch('http://localhost:9200')

In [6]:
client.info()

ObjectApiResponse({'name': 'eb363035b330', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'FaccVoeSS5e6WMqIialyXQ', 'version': {'number': '8.17.6', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'dbcbbbd0bc4924cfeb28929dc05d82d662c527b7', 'build_date': '2025-04-30T14:07:12.231372970Z', 'build_snapshot': False, 'lucene_version': '9.12.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [7]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}

In [8]:
index_name = "course-questions"
client.indices.create(index=index_name, body=index_settings)

BadRequestError: BadRequestError(400, 'resource_already_exists_exception', 'index [course-questions/_9RDQJRiTaqYZI9oeXHVig] already exists')

In [12]:
from tqdm import tqdm

In [13]:
for doc in tqdm(documents):
    client.index(index=index_name, document=doc)

100%|██████████| 948/948 [00:04<00:00, 202.98it/s]


## Q3. Searching

In [14]:
query = "How do execute a command on a Kubernetes pod?"

In [21]:
search_query = {
    "size": 4,
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question^4", "text"],
                    "type": "best_fields"
                }
            },
        }
    }
}

search_results = client.search(index=index_name, body=search_query)

In [22]:
search_results['hits']['hits'][0]['_score']

44.50556

## Q4. Filtering

In [27]:
query2 = "How do copy a file to a Docker container?"

In [28]:
search_query2 = {
    "size": 3,
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query2,
                    "fields": ["question^4", "text"],
                    "type": "best_fields"
                }
            },
            "filter": {
                "term": {
                    "course": "machine-learning-zoomcamp"
                }
            }
        }
    }
}

search_results2 = client.search(index=index_name, body=search_query)

In [30]:
search_results2['hits']['hits'][2]

{'_index': 'course-questions',
 '_id': '4OJ3gpcBA5JPcOLU_tcg',
 '_score': 33.70974,
 '_source': {'text': 'You can copy files from your local machine into a Docker container using the docker cp command. Here\'s how to do it:\nIn the Dockerfile, you can provide the folder containing the files that you want to copy over. The basic syntax is as follows:\nCOPY ["src/predict.py", "models/xgb_model.bin", "./"]\t\t\t\t\t\t\t\t\t\t\tGopakumar Gopinathan',
  'section': '5. Deploying Machine Learning Models',
  'question': 'How do I copy files from a different folder into docker container’s working directory?',
  'course': 'machine-learning-zoomcamp'}}

## Q5. Building a prompt

In [33]:
context_template = """
Q: {question}
A: {text}
""".strip()

prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

In [45]:
results = []

for hit in search_results2['hits']['hits']:
    doc = hit['_source']
    context_piece = context_template.format(**doc)
    results.append(context_piece)

context = '\n\n'.join(results)

In [60]:
prompt = prompt_template.format(question=query2, context=context)
len(prompt)

1434

In [ ]:
!pip install tiktoken

## Q6. Tokens

In [63]:
import tiktoken
encoding = tiktoken.encoding_for_model("gpt-4o")
len(encoding.encode(prompt))

320